#### **Simplified Attention Mechanism without trainable Weights**

In [1]:
### Without Trainable Weights means it's tells about the semantic meaning between the tokens. 
##suppose mat is warm, warm is not similar to the mat , so it will pay attention to the mat
## But in real scenario based sentence warm must pay attention to the mat because warm tells about mat.

In [2]:
import torch


inputs = torch.tensor([
    [0.8, 0.1, 0.2],   #← Your
    [1.0, 0.7, 0.8],   #← Journey
    [0.7, 1.0, 0.3],   #← Starts
    [0.2, 0.2, 0.6],   #← with
    [0.3, 0.4, 0.9],   #← one
    [0.8, 0.9, 1.0]    #← step
])

In [3]:
torch.empty(inputs.shape[0])

tensor([-9.5511e-38,  1.9730e-42,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00])

In [4]:
## using the dot product we can calculate the how two vectors are aligned

query = inputs[1] ## 2nd token as query, using we can calculate across all the vectors using dot


attention_scores = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
    attention_scores[i] = torch.dot(x_i,query)

print(attention_scores)

tensor([1.0300, 2.1300, 1.6400, 0.8200, 1.3000, 2.2300])


#### **Normalization means converts the raw similarity scores into a stable distribution of attention weights.**

In [5]:
### generic way

attention_weights = attention_scores/attention_scores.sum()

print("Attention Weights:",attention_weights)
print("Sum of the Weights:",attention_weights.sum())

Attention Weights: tensor([0.1126, 0.2328, 0.1792, 0.0896, 0.1421, 0.2437])
Sum of the Weights: tensor(1.0000)


In [6]:
## let's implement the general softmax to normalize the scores

def soft_naive(x):
    return torch.exp(x)/torch.exp(x).sum(dim=0)

attention_weights_naive = soft_naive(attention_scores)

print("Navie Attention Weights:",attention_weights_naive)

print("Sum:",attention_weights_naive.sum(dim=0))

Navie Attention Weights: tensor([0.0886, 0.2662, 0.1631, 0.0718, 0.1161, 0.2942])
Sum: tensor(1.)


In [7]:
### lets implement the pytorch softmax by subtract the max value. 
attention_weights_torch = torch.softmax(attention_scores,dim = 0)

print("Attention Weights Using Pytorch Softmax:",attention_weights_torch)
print("Sum:",attention_weights_torch.sum(dim=0))

Attention Weights Using Pytorch Softmax: tensor([0.0886, 0.2662, 0.1631, 0.0718, 0.1161, 0.2942])
Sum: tensor(1.)


In [8]:
## see both softmax and pytorch softmax weights are same because of very less value. 
### Pytorch softmax solves the issues like underflow and overflow for larger and lowe values. 

In [9]:
#context vector for the word journey

query = inputs[1] # journey

context_vector = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):

    context_vector+=attention_weights_torch[i]*x_i

print(context_vector)


tensor([0.7358, 0.6839, 0.7214])


In [10]:
### lets calculate the attention scores all words in input

final_attention_scores = torch.empty(6,6)
for i,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        final_attention_scores[i,j] = torch.dot(x_i,x_j)
print(final_attention_scores)

tensor([[0.6900, 1.0300, 0.7200, 0.3000, 0.4600, 0.9300],
        [1.0300, 2.1300, 1.6400, 0.8200, 1.3000, 2.2300],
        [0.7200, 1.6400, 1.5800, 0.5200, 0.8800, 1.7600],
        [0.3000, 0.8200, 0.5200, 0.4400, 0.6800, 0.9400],
        [0.4600, 1.3000, 0.8800, 0.6800, 1.0600, 1.5000],
        [0.9300, 2.2300, 1.7600, 0.9400, 1.5000, 2.4500]])


In [11]:
## more simpler way and also loop's are very slow
final_attn_scores = inputs @ inputs.T
print(final_attn_scores)

tensor([[0.6900, 1.0300, 0.7200, 0.3000, 0.4600, 0.9300],
        [1.0300, 2.1300, 1.6400, 0.8200, 1.3000, 2.2300],
        [0.7200, 1.6400, 1.5800, 0.5200, 0.8800, 1.7600],
        [0.3000, 0.8200, 0.5200, 0.4400, 0.6800, 0.9400],
        [0.4600, 1.3000, 0.8800, 0.6800, 1.0600, 1.5000],
        [0.9300, 2.2300, 1.7600, 0.9400, 1.5000, 2.4500]])


In [12]:
final_attention_weights = torch.softmax(final_attention_scores,dim=1)
print(final_attention_weights)
print(final_attention_weights.sum(dim=1))

tensor([[0.1619, 0.2274, 0.1668, 0.1096, 0.1286, 0.2058],
        [0.0886, 0.2662, 0.1631, 0.0718, 0.1161, 0.2942],
        [0.0935, 0.2346, 0.2210, 0.0766, 0.1097, 0.2646],
        [0.1185, 0.1994, 0.1477, 0.1363, 0.1733, 0.2248],
        [0.0931, 0.2158, 0.1418, 0.1161, 0.1697, 0.2635],
        [0.0699, 0.2564, 0.1602, 0.0706, 0.1235, 0.3194]])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [13]:
final_context_vectors = final_attention_weights @ inputs
print(final_context_vectors)

tensor([[0.6987, 0.6007, 0.6516],
        [0.7358, 0.6839, 0.7214],
        [0.7240, 0.6919, 0.6820],
        [0.6566, 0.5980, 0.6901],
        [0.6745, 0.6304, 0.7197],
        [0.7311, 0.6977, 0.7401]])


In [14]:
context_vector

tensor([0.7358, 0.6839, 0.7214])